In [1]:
import sys; sys.path.insert(0, "../src")
from ds_prep.stats_utils import srm_check

import numpy as np
import math
from scipy.stats import chisquare

In [2]:
def compare(name, actual, expected, rel_tol=1e-9):
    """
    Check whether `actual` matches `expected` and return a labelled tick or cross.

    name : str
        Label shown in the output.
    actual : bool or float
        Value from your own implementation.
    expected : bool or float
        Reference value (e.g. from scipy).
    rel_tol : float
        Relative tolerance for numeric comparisons.
    """
    # bool must be checked first: in Python, bool is a subclass of int
    if isinstance(actual, (bool, np.bool_)) or isinstance(expected, (bool, np.bool_)):
        ok = bool(actual) == bool(expected)
    else:
        ok = math.isclose(float(actual), float(expected), rel_tol=rel_tol, abs_tol=0.0)

    return f"{name} {'✅' if ok else '❌'}"

In [3]:
# Q1. A 90/10 ramp ends with 9,120 control and 880 treatment. Is this SRM?
observed =  np.asarray([9120, 880])
expected_ratios =  np.asarray([0.5, 0.5])
expected = observed.sum() * expected_ratios
alpha = 0.001

chi2_act, p_value_act = chisquare(observed, expected)
is_srm_act = p_value_act < alpha

pred_result = srm_check(observed, expected_ratios, alpha)
chi2_pred = pred_result["chi2"]
p_value_pred = pred_result["p_value"]
is_srm_pred = pred_result["srm_detected"]

print(f"{compare("chi2", chi2_act, chi2_pred)}, actual: {chi2_act:.4f}, predicted: {chi2_pred:.4f}")
print(f"{compare("p_value", p_value_act, p_value_pred)}, actual: {p_value_act:.4f}, predicted: {p_value_pred:.4f}")
print(f"{compare("is_srm", is_srm_act, is_srm_pred)}, actual: {is_srm_act}, predicted: {is_srm_pred}")

chi2 ✅, actual: 6789.7600, predicted: 6789.7600
p_value ✅, actual: 0.0000, predicted: 0.0000
is_srm ✅, actual: True, predicted: True


Q2. Treatment adds a redirect that drops about 2% of users before the exposure event is logged. Which arm comes out smaller? Why can’t you fix the effect estimate by “just reweighting the counts”?

Treatment arm would come out smaller than expected. Effect estimate cannot be fixed just by reweighting is because there is no certain way to figure out whether drop happened randomly or due to a specific reason. And real world crash mostly targets a specific set of condition matching users so we are never able to perform our test on those users

In [4]:
# Q3.1 A 3-arm equal split ends with 3,410 / 3,350 / 3,240. 
observed =  np.asarray([3410, 3350, 3240])
expected_ratios =  np.asarray([1/3, 1/3, 1/3])
expected = observed.sum() * expected_ratios
alpha = 0.001

chi2_act, p_value_act = chisquare(observed, expected)
is_srm_act = p_value_act < alpha

pred_result = srm_check(observed, expected_ratios, alpha)
chi2_pred = pred_result["chi2"]
p_value_pred = pred_result["p_value"]
is_srm_pred = pred_result["srm_detected"]

print(f"{compare("chi2", chi2_act, chi2_pred)}, actual: {chi2_act:.4f}, predicted: {chi2_pred:.4f}")
print(f"{compare("p_value", p_value_act, p_value_pred)}, actual: {p_value_act:.4f}, predicted: {p_value_pred:.4f}")
print(f"{compare("is_srm", is_srm_act, is_srm_pred)}, actual: {is_srm_act}, predicted: {is_srm_pred}")

chi2 ✅, actual: 4.4600, predicted: 4.4600
p_value ✅, actual: 0.1075, predicted: 0.1075
is_srm ✅, actual: False, predicted: False


Q3.2 Is this SRM? What df did you use, and why?

This isn't srm and df used was 2. Chi2 follows a different distribution with degree of freedom = number of arms - 1. In case of two arms it is same as z2 but higher arms have fatter distribution